## 0. Khởi tạo môi trường


In [1]:
import os
import sys
import re
import math
import sqlite3
import shutil
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import orjson
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, f1_score, mean_absolute_error
from sklearn.model_selection import StratifiedShuffleSplit

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

for folder in ['data/chart_img', 'data/df', 'data/dataset']:
    Path(folder).mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]} | PyArrow: {pa.__version__} | NumPy: {np.__version__} | Pandas: {pd.__version__}")
print("Khởi tạo môi trường thành công!")

Python: 3.11.18 | PyArrow: 17.0.0 | NumPy: 1.26.4 | Pandas: 3.0.6
Khởi tạo môi trường thành công!


## 1. Import dataset


In [2]:
json_candidates = [
    Path("data/dataset/TripAdvisor_EN.json"),
    Path("/kaggle/input/dap_dts/TripAdvisor_EN.json"),
    Path("/kaggle/input/dap-dts/TripAdvisor_EN.json"),
    Path("TripAdvisor_EN.json")
]
dataset_path = next((p for p in json_candidates if p.exists()), None)
if dataset_path is None and Path("/kaggle/input").exists():
    found_jsons = list(Path("/kaggle/input").rglob("TripAdvisor_EN.json"))
    if found_jsons:
        dataset_path = found_jsons[0]
if dataset_path is None:
    dataset_path = Path("data/dataset/TripAdvisor_EN.json")

with open(dataset_path, "rb") as f:
    raw_json = orjson.loads(f.read())
print(f"Đã đọc {len(raw_json):,} bản ghi bằng orjson.")

reviews_list = []
annotations_list = []

for idx, item in enumerate(raw_json, 1):
    review_id = idx
    raw_item_id = item.get("id")
    data_block = item.get("data", {})
    raw_text = unicodedata.normalize("NFC", data_block.get("text", "").strip())
    meta = data_block.get("meta_info", {})

    pos_c, neg_c, neu_c = 0, 0, 0
    ann_container = item.get("annotations", [])
    if ann_container:
        results = ann_container[0].get("result", [])
        entities_dict = {}
        sentiments_dict = {}

        for res in results:
            res_id = res.get("id")
            fn = res.get("from_name")
            val = res.get("value", {})
            lbls = val.get("labels", [])
            if not lbls:
                continue
            if fn == "entities":
                entities_dict[res_id] = (lbls[0], int(val.get("start", 0)), int(val.get("end", 0)), val.get("text", ""))
            elif fn == "entity_sentiment":
                sentiments_dict[res_id] = lbls[0]

        for res_id, (aspect, start_c, end_c, span_txt) in entities_dict.items():
            sentiment = sentiments_dict.get(res_id, "Neutral")
            if sentiment == "Positive":
                pos_c += 1
            elif sentiment == "Negative":
                neg_c += 1
            else:
                neu_c += 1
            annotations_list.append({
                "annotation_id": f"{review_id}_{res_id}",
                "review_id": review_id,
                "aspect": aspect,
                "sentiment": sentiment,
                "start_char": start_c,
                "end_char": end_c,
                "span_text": span_txt.strip()
            })

    if pos_c > neg_c and pos_c >= neu_c:
        overall_sentiment, sentiment_id = "Positive", 2
    elif neg_c > pos_c and neg_c >= neu_c:
        overall_sentiment, sentiment_id = "Negative", 0
    elif pos_c > neg_c:
        overall_sentiment, sentiment_id = "Positive", 2
    elif neg_c > pos_c:
        overall_sentiment, sentiment_id = "Negative", 0
    else:
        overall_sentiment, sentiment_id = "Neutral", 1

    reviews_list.append({
        "review_id": review_id,
        "task_id": raw_item_id,
        "ref_id": meta.get("ref_id", f"TripAdvisor_En_{review_id}"),
        "hotel_name": meta.get("hotel name", "").strip(),
        "hotel_address": meta.get("hotel address", "").strip(),
        "location": meta.get("location", "").strip(),
        "star": meta.get("star", "").strip(),
        "overall_sentiment": overall_sentiment,
        "sentiment_id": sentiment_id,
        "date": meta.get("date", "").strip(),
        "year": int(meta.get("year", 0)) if str(meta.get("year", "")).isdigit() else None,
        "month": int(meta.get("month", 0)) if str(meta.get("month", "")).isdigit() else None,
        "review_text": raw_text,
        "char_length": len(raw_text),
        "word_count": len(raw_text.split())
    })

df_reviews = pd.DataFrame(reviews_list)
df_annotations = pd.DataFrame(annotations_list)

print(f"Tổng số reviews: {len(df_reviews):,} | Tổng số aspect annotations: {len(df_annotations):,}")

Đã đọc 9,990 bản ghi bằng orjson.
Tổng số reviews: 9,990 | Tổng số aspect annotations: 54,326


## 2. Dataframe và SQL Queries


### 2.1 Khởi tạo Dataframe


In [3]:
os.makedirs("data/df", exist_ok=True)
db_path = "data/df/chatbot.db"

if Path("chatbot.db").exists():
    Path("chatbot.db").unlink()

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS reviews;")
cursor.execute("DROP TABLE IF EXISTS aspect_annotations;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS reviews (
    review_id INTEGER PRIMARY KEY,
    task_id INTEGER,
    ref_id TEXT,
    hotel_name TEXT,
    hotel_address TEXT,
    location TEXT,
    star TEXT,
    overall_sentiment TEXT,
    sentiment_id INTEGER,
    date TEXT,
    year INTEGER,
    month INTEGER,
    review_text TEXT,
    char_length INTEGER,
    word_count INTEGER
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS aspect_annotations (
    annotation_id TEXT PRIMARY KEY,
    review_id INTEGER,
    aspect TEXT,
    sentiment TEXT,
    start_char INTEGER,
    end_char INTEGER,
    span_text TEXT,
    FOREIGN KEY (review_id) REFERENCES reviews(review_id)
);
""")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_sentiment ON reviews(sentiment_id, overall_sentiment);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_location ON reviews(location);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ann_aspect_sentiment ON aspect_annotations(aspect, sentiment);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ann_review_id ON aspect_annotations(review_id);")

df_reviews.to_sql("reviews", conn, if_exists="replace", index=False)
df_annotations.to_sql("aspect_annotations", conn, if_exists="replace", index=False)
conn.commit()

r_cnt = cursor.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
a_cnt = cursor.execute("SELECT COUNT(*) FROM aspect_annotations").fetchone()[0]
print(f"Khởi tạo CSDL tại '{db_path}' thành công! Số reviews: {r_cnt:,} | Số annotations: {a_cnt:,}")

Khởi tạo CSDL tại 'data/df/chatbot.db' thành công! Số reviews: 9,990 | Số annotations: 54,326


### 2.2 SQL Queries cho các Research Questions (RQs)


In [4]:
query_1 = """
SELECT
    sentiment_id,
    overall_sentiment,
    COUNT(review_id) AS total_reviews,
    ROUND(100.0 * COUNT(review_id) / (SELECT COUNT(*) FROM reviews), 2) AS pct_share
FROM reviews
GROUP BY sentiment_id, overall_sentiment
ORDER BY sentiment_id ASC;
"""
df_q1 = pd.read_sql_query(query_1, conn)
print("Kết quả Query 1 - RQ1:")
display(df_q1)

Kết quả Query 1 - RQ1:


,sentiment_id,overall_sentiment,total_reviews,pct_share
0,0,Negative,668,6.69
1,1,Neutral,314,3.14
2,2,Positive,9008,90.17


In [5]:
query_2 = """
SELECT
    r.sentiment_id,
    r.overall_sentiment,
    COUNT(a.annotation_id) AS total_rationales,
    ROUND(AVG(a.end_char - a.start_char), 2) AS avg_rationale_chars,
    ROUND(AVG(CAST(a.end_char - a.start_char AS REAL) / r.char_length) * 100, 2) AS avg_evidence_coverage_pct
FROM reviews r
JOIN aspect_annotations a ON r.review_id = a.review_id
WHERE r.char_length > 0
GROUP BY r.sentiment_id, r.overall_sentiment
ORDER BY r.sentiment_id ASC;
"""
df_q2 = pd.read_sql_query(query_2, conn)
print("Kết quả Query 2 - RQ1:")
display(df_q2)

Kết quả Query 2 - RQ1:


,sentiment_id,overall_sentiment,total_rationales,avg_rationale_chars,avg_evidence_coverage_pct
0,0,Negative,3506,72.29,14.00
1,1,Neutral,910,72.05,15.02
2,2,Positive,49910,62.81,14.53


In [6]:
query_3 = """
SELECT
    a1.aspect AS aspect_x,
    a2.aspect AS aspect_y,
    COUNT(DISTINCT a1.review_id) AS co_occurrence_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.aspect < a2.aspect
GROUP BY aspect_x, aspect_y
ORDER BY co_occurrence_count DESC;
"""
df_q3 = pd.read_sql_query(query_3, conn)
print("Kết quả Query 3 - RQ2:")
display(df_q3)

Kết quả Query 3 - RQ2:


,aspect_x,aspect_y,co_occurrence_count
0,Facility,Service,6932
1,Experience,Service,5512
2,Amenity,Service,5224
3,Amenity,Facility,4496
4,Experience,Facility,4483
5,Loyalty,Service,3717
6,Amenity,Experience,3364
7,Facility,Loyalty,2873
8,Experience,Loyalty,2414
9,Amenity,Loyalty,2181


In [7]:
query_4 = """
SELECT
    a1.aspect AS aspect_pos,
    a2.aspect AS aspect_neg,
    COUNT(DISTINCT a1.review_id) AS conflicting_reviews_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.aspect != a2.aspect
WHERE a1.sentiment = 'Positive' AND a2.sentiment = 'Negative'
GROUP BY aspect_pos, aspect_neg
ORDER BY conflicting_reviews_count DESC
LIMIT 10;
"""
df_q4 = pd.read_sql_query(query_4, conn)
print("Kết quả Query 4 - RQ2:")
display(df_q4)

Kết quả Query 4 - RQ2:


,aspect_pos,aspect_neg,conflicting_reviews_count
0,Service,Facility,640
1,Amenity,Facility,480
2,Facility,Service,384
3,Service,Experience,359
4,Experience,Facility,337
5,Facility,Experience,316
6,Amenity,Service,281
7,Amenity,Experience,246
8,Experience,Service,203
9,Service,Amenity,187


In [8]:
query_5 = """
SELECT
    a1.aspect AS aspect_1,
    a2.aspect AS aspect_2,
    COUNT(*) AS collision_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.annotation_id < a2.annotation_id
WHERE a1.aspect != a2.aspect
  AND NOT (a1.end_char <= a2.start_char OR a1.start_char >= a2.end_char)
GROUP BY aspect_1, aspect_2
ORDER BY collision_count DESC;
"""
df_q5 = pd.read_sql_query(query_5, conn)
print("Kết quả Query 5 - RQ3:")
display(df_q5)

Kết quả Query 5 - RQ3:


,aspect_1,aspect_2,collision_count
0,Facility,Service,2
1,Amenity,Facility,1
2,Experience,Amenity,1
3,Facility,Experience,1
4,Loyalty,Amenity,1
5,Service,Experience,1
6,Service,Loyalty,1


In [9]:
query_6 = """
SELECT
    r.sentiment_id,
    r.overall_sentiment,
    a.aspect,
    COUNT(DISTINCT r.review_id) AS review_count,
    SUM(CASE WHEN a.sentiment = 'Positive' THEN 1 ELSE 0 END) AS pos_cnt,
    SUM(CASE WHEN a.sentiment = 'Neutral' THEN 1 ELSE 0 END) AS neu_cnt,
    SUM(CASE WHEN a.sentiment = 'Negative' THEN 1 ELSE 0 END) AS neg_cnt,
    ROUND(CAST(COUNT(r.review_id) AS REAL) / (SELECT COUNT(*) FROM reviews), 4) AS class_prior
FROM reviews r
JOIN aspect_annotations a ON r.review_id = a.review_id
WHERE a.aspect IN ('Service', 'Facility', 'Experience')
GROUP BY r.sentiment_id, r.overall_sentiment, a.aspect
ORDER BY a.aspect, r.sentiment_id ASC;
"""
df_q6 = pd.read_sql_query(query_6, conn)
print("Kết quả Query 6 - RQ3:")
display(df_q6)

Kết quả Query 6 - RQ3:


,sentiment_id,overall_sentiment,aspect,review_count,pos_cnt,neu_cnt,neg_cnt,class_prior
0,0,Negative,Experience,385,58,58,494,0.0611
1,1,Neutral,Experience,100,46,32,72,0.0150
2,2,Positive,Experience,5544,7627,420,306,0.8361
3,0,Negative,Facility,503,182,133,830,0.1146
4,1,Neutral,Facility,155,121,56,140,0.0317
5,2,Positive,Facility,6801,9585,936,658,1.1190
6,0,Negative,Service,539,166,106,799,0.1072
7,1,Neutral,Service,152,121,47,99,0.0267
8,2,Positive,Service,8437,16988,632,331,1.7969


## 3. Data Cleaning và EDA


### 3.1 Kiểm định tính toàn vẹn dữ liệu và kiểm tra Missing Values


In [10]:
missing_reviews = df_reviews.isnull().sum()
missing_ann = df_annotations.isnull().sum()
print("Missing values trong df_reviews:\n", missing_reviews[missing_reviews > 0])
print("Missing values trong df_annotations:\n", missing_ann[missing_ann > 0])

invalid_spans = df_annotations[
    (df_annotations['start_char'] < 0) |
    (df_annotations['start_char'] >= df_annotations['end_char'])
]
print(f"Số lượng annotation spans có tọa độ không hợp lệ: {len(invalid_spans)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

completeness = (1 - df_reviews[['hotel_name', 'location', 'star', 'overall_sentiment', 'date', 'review_text']].isnull().mean()) * 100
sns.barplot(x=completeness.values, y=completeness.index, ax=axes[0], color='#2b5c8f')
axes[0].set_title("Data Completeness Rate (%)", fontweight='bold')
axes[0].set_xlim(0, 105)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.1f}%", (p.get_width() + 1, p.get_y() + p.get_height() / 2), va='center')

sns.histplot(df_reviews['word_count'], bins=50, kde=True, ax=axes[1], color='#e67e22')
axes[1].axvline(df_reviews['word_count'].median(), color='red', linestyle='--', label=f"Median: {df_reviews['word_count'].median():.0f} words")
axes[1].set_title("Distribution of Review Word Counts", fontweight='bold')
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
chart_path_31 = "data/chart_img/31_missing_values_and_data_summary.png"
plt.savefig(chart_path_31)
plt.close()
print(f"Đã lưu biểu đồ làm sạch dữ liệu tại: '{chart_path_31}'")

Missing values trong df_reviews:
 Series([], dtype: int64)
Missing values trong df_annotations:
 Series([], dtype: int64)
Số lượng annotation spans có tọa độ không hợp lệ: 0
Đã lưu biểu đồ làm sạch dữ liệu tại: 'data/chart_img/31_missing_values_and_data_summary.png'


### 3.2 Thống kê mô tả đặc tính văn bản và phân vị độ dài


In [11]:
desc_stats = df_reviews[['char_length', 'word_count', 'sentiment_id']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
print("Bảng thống kê mô tả đặc tính văn bản đánh giá:")
display(desc_stats)

plt.figure(figsize=(10, 5))
sns.kdeplot(df_reviews['char_length'], fill=True, color='#27ae60', alpha=0.4)
plt.title("Kernel Density Estimation of Character Lengths", fontweight='bold')
plt.xlabel("Review Character Length")
plt.ylabel("Density")
chart_path_32 = "data/chart_img/32_review_text_length_distribution.png"
plt.savefig(chart_path_32)
plt.close()
print(f"Đã lưu biểu đồ phân bố độ dài tại: '{chart_path_32}'")

Bảng thống kê mô tả đặc tính văn bản đánh giá:
Đã lưu biểu đồ phân bố độ dài tại: 'data/chart_img/32_review_text_length_distribution.png'


,count,mean,std,min,5%,25%,50%,75%,95%,max
char_length,9990.0,450.664464,200.472604,177.0,210.45,274.0,398.0,618.0,786.0,847.0
word_count,9990.0,82.145846,37.112671,30.0,37.00,49.0,72.0,114.0,146.0,150.0
sentiment_id,9990.0,1.834835,0.521197,0.0,0.00,2.0,2.0,2.0,2.0,2.0


## 4. Data Visualization


### 4.1 Phân bố cảm xúc tổng quan 3 lớp (Negative, Neutral, Positive)


In [12]:
plt.figure(figsize=(8, 5))
sent_order = ['Negative', 'Neutral', 'Positive']
sent_counts = df_reviews['overall_sentiment'].value_counts().reindex(sent_order, fill_value=0)
bars = plt.bar(sent_counts.index, sent_counts.values, color=['#e74c3c', '#f1c40f', '#2ecc71'], width=0.55)
plt.title("4.1 Phân bố cảm xúc tổng quan 3 lớp (3-Class Sentiment Distribution)", fontweight='bold')
plt.xlabel("Nhãn cảm xúc tổng quan (Negative, Neutral, Positive)")
plt.ylabel("Số lượng đánh giá")
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df_reviews)) * 100
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f"{yval:,}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10)
plt.ylim(0, max(sent_counts.values) * 1.15)
plt.tight_layout()
chart_path_41 = "data/chart_img/41_overall_rating_distribution.png"
plt.savefig(chart_path_41)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_41}'")

Đã lưu biểu đồ: 'data/chart_img/41_overall_rating_distribution.png'


### 4.2 Top 10 khách sạn có số review cao nhất


In [13]:
plt.figure(figsize=(12, 6))
top_hotels = df_reviews["hotel_name"].value_counts().head(10)
hotel_labels = [unicodedata.normalize("NFC", name).strip().title() for name in top_hotels.index]

hotel_pos_ratio = df_reviews.groupby("hotel_name")["sentiment_id"].apply(lambda s: (s == 2).mean() * 100)

sns.barplot(x=top_hotels.values, y=hotel_labels, palette="Blues_r", hue=hotel_labels, legend=False)
plt.title("4.2 Top 10 Khách sạn có review cao nhất", fontweight="bold", fontsize=13, pad=12)
plt.xlabel("Số lượng bài đánh giá", fontsize=11)
plt.ylabel("Khách sạn", fontsize=11)

for i, (orig_name, v) in enumerate(top_hotels.items()):
    pos_r = hotel_pos_ratio.get(orig_name, 0.0)
    pct = (v / len(df_reviews)) * 100
    plt.text(v + 1, i, f"{v:,} ({pct:.1f}%) | Tỷ lệ Positive: {pos_r:.1f}%", va="center", fontsize=9.5)

plt.xlim(0, max(top_hotels.values) * 1.38)
plt.tight_layout()
chart_path_42 = "data/chart_img/42_top10_hotels_highest_reviews.png"
plt.savefig(chart_path_42, dpi=300)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_42}'")

Đã lưu biểu đồ: 'data/chart_img/42_top10_hotels_highest_reviews.png'


### 4.3 Tần suất xuất hiện của các khía cạnh khách sạn


In [14]:
plt.figure(figsize=(9, 5))
aspect_counts = df_annotations['aspect'].value_counts()
sns.barplot(x=aspect_counts.index, y=aspect_counts.values, palette='viridis', hue=aspect_counts.index, legend=False)
plt.title("4.3 Tần suất xuất hiện của các khía cạnh khách sạn", fontweight='bold')
plt.xlabel("Khía cạnh (Aspect)")
plt.ylabel("Số lượng đoạn bằng chứng (Annotations)")
for i, v in enumerate(aspect_counts.values):
    plt.text(i, v + 300, f"{v:,}", ha='center', fontsize=10, fontweight='bold')
plt.ylim(0, max(aspect_counts.values) * 1.12)
plt.tight_layout()
chart_path_43 = "data/chart_img/43_aspect_frequency_distribution.png"
plt.savefig(chart_path_43)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_43}'")

Đã lưu biểu đồ: 'data/chart_img/43_aspect_frequency_distribution.png'


### 4.4 Phân bố cảm xúc theo từng khía cạnh


In [15]:
asp_sent = pd.crosstab(df_annotations['aspect'], df_annotations['sentiment'], normalize='index') * 100
cols = [c for c in ['Positive', 'Neutral', 'Negative'] if c in asp_sent.columns]
asp_sent = asp_sent[cols]
colors = {'Positive': '#2ecc71', 'Neutral': '#f1c40f', 'Negative': '#e74c3c'}
ax = asp_sent.plot(kind='bar', stacked=True, figsize=(10, 5), color=[colors[c] for c in cols], edgecolor='none')
plt.title("4.4 Tỷ lệ cảm xúc theo từng khía cạnh khách sạn (%)", fontweight='bold')
plt.xlabel("Khía cạnh (Aspect)")
plt.ylabel("Tỷ lệ phần trăm (%)")
plt.legend(title="Cảm xúc", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=15)
plt.ylim(0, 100)
plt.tight_layout()
chart_path_44 = "data/chart_img/44_aspect_sentiment_breakdown.png"
plt.savefig(chart_path_44)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_44}'")

Đã lưu biểu đồ: 'data/chart_img/44_aspect_sentiment_breakdown.png'


### 4.5 Độ dài đoạn bằng chứng theo khía cạnh


In [16]:
df_annotations['span_len'] = df_annotations['end_char'] - df_annotations['start_char']
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_annotations, x='aspect', y='span_len', palette='Set2', hue='aspect', legend=False, showfliers=False)
plt.title("4.5 Phân bố độ dài ký tự của đoạn bằng chứng (Rationale Span Length)", fontweight='bold')
plt.xlabel("Khía cạnh (Aspect)")
plt.ylabel("Độ dài đoạn văn (Ký tự)")
plt.tight_layout()
chart_path_45 = "data/chart_img/45_rationale_length_by_aspect.png"
plt.savefig(chart_path_45)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_45}'")

Đã lưu biểu đồ: 'data/chart_img/45_rationale_length_by_aspect.png'


### 4.6 Ma trận tương quan đồng xuất hiện giữa các khía cạnh


In [17]:
aspect_list = sorted(df_annotations['aspect'].unique())
co_mat = pd.DataFrame(0, index=aspect_list, columns=aspect_list)

rev_aspects = df_annotations.groupby('review_id')['aspect'].unique()
for asps in rev_aspects:
    for a1 in asps:
        for a2 in asps:
            co_mat.loc[a1, a2] += 1

plt.figure(figsize=(8, 6))
sns.heatmap(co_mat, annot=True, fmt='d', cmap='YlGnBu', cbar=True)
plt.title("4.6 Ma trận đồng xuất hiện giữa các khía cạnh trong cùng review", fontweight='bold')
plt.xlabel("Khía cạnh")
plt.ylabel("Khía cạnh")
plt.tight_layout()
chart_path_46 = "data/chart_img/46_aspect_cooccurrence_matrix.png"
plt.savefig(chart_path_46)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_46}'")

Đã lưu biểu đồ: 'data/chart_img/46_aspect_cooccurrence_matrix.png'


### 4.7 Tỷ lệ phân kỳ cảm xúc giữa các cặp khía cạnh trong cùng review


In [18]:
conflict_mat = pd.DataFrame(0, index=aspect_list, columns=aspect_list)
rev_asp_sent = df_annotations.groupby(['review_id', 'aspect'])['sentiment'].unique()

rev_to_asp = defaultdict(dict)
for (rid, asp), sents in rev_asp_sent.items():
    rev_to_asp[rid][asp] = sents

for rid, asps_dict in rev_to_asp.items():
    for a1, s1 in asps_dict.items():
        for a2, s2 in asps_dict.items():
            if a1 != a2 and 'Positive' in s1 and 'Negative' in s2:
                conflict_mat.loc[a1, a2] += 1

plt.figure(figsize=(8, 6))
sns.heatmap(conflict_mat, annot=True, fmt='d', cmap='Reds', cbar=True)
plt.title("4.7 Ma trận phân kỳ cảm xúc (Aspect A Pos vs Aspect B Neg)", fontweight='bold')
plt.xlabel("Khía cạnh tiêu cực (Negative)")
plt.ylabel("Khía cạnh tích cực (Positive)")
plt.tight_layout()
chart_path_47 = "data/chart_img/47_sentiment_divergence_rate.png"
plt.savefig(chart_path_47)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_47}'")

Đã lưu biểu đồ: 'data/chart_img/47_sentiment_divergence_rate.png'


### 4.8 Phân bố số lượng từ của bài đánh giá theo nhãn cảm xúc 3 lớp


In [19]:
plt.figure(figsize=(9, 5))
sent_order = ['Negative', 'Neutral', 'Positive']
sns.boxplot(
    data=df_reviews, x='overall_sentiment', y='word_count',
    order=sent_order, hue='overall_sentiment', hue_order=sent_order,
    palette=['#e74c3c', '#f1c40f', '#2ecc71'], legend=False, showfliers=False
)
plt.title("4.8 Phân bố độ dài từ (Word Count) theo nhãn cảm xúc 3 lớp", fontweight='bold')
plt.xlabel("Nhãn cảm xúc (Negative, Neutral, Positive)")
plt.ylabel("Số từ trong bài đánh giá")
plt.tight_layout()
chart_path_48 = "data/chart_img/48_review_wordcount_by_star.png"
plt.savefig(chart_path_48)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_48}'")

Đã lưu biểu đồ: 'data/chart_img/48_review_wordcount_by_star.png'


### 4.9 Mức độ chồng lấn tọa độ văn bản giữa các khía cạnh


In [20]:
collision_records = []
ann_grouped = df_annotations.groupby('review_id')

for rid, group in ann_grouped:
    records = group[['annotation_id', 'aspect', 'start_char', 'end_char']].to_dict('records')
    n = len(records)
    for i in range(n):
        for j in range(i + 1, n):
            r1, r2 = records[i], records[j]
            if r1['aspect'] != r2['aspect']:
                if not (r1['end_char'] <= r2['start_char'] or r1['start_char'] >= r2['end_char']):
                    pair_label = f"{min(r1['aspect'], r2['aspect'])} & {max(r1['aspect'], r2['aspect'])}"
                    collision_records.append(pair_label)

plt.figure(figsize=(8, 4.5))
if collision_records:
    col_counts = pd.Series(collision_records).value_counts()
    sns.barplot(x=col_counts.values, y=col_counts.index, color='#c0392b')
    plt.title("4.9 Số trường hợp chồng lấn tọa độ ký tự giữa các khía cạnh", fontweight='bold')
    plt.xlabel("Số vụ va chạm tọa độ văn bản")
    for i, v in enumerate(col_counts.values):
        plt.text(v + 0.1, i, str(v), va='center', fontweight='bold')
    plt.xlim(0, max(col_counts.values) + 1)
else:
    plt.text(0.5, 0.5, "Không phát hiện va chạm tọa độ", ha='center', va='center')
    plt.title("4.9 Số trường hợp chồng lấn tọa độ giữa các khía cạnh", fontweight='bold')
plt.tight_layout()
chart_path_49 = "data/chart_img/49_cross_aspect_span_overlap.png"
plt.savefig(chart_path_49)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_49}'")

Đã lưu biểu đồ: 'data/chart_img/49_cross_aspect_span_overlap.png'


### 4.10 Phân bố số lượng khía cạnh được đề cập trên mỗi bài đánh giá


In [21]:
aspects_per_rev = df_annotations.groupby('review_id')['aspect'].nunique()
full_aspects_per_rev = df_reviews['review_id'].map(aspects_per_rev).fillna(0).astype(int)

plt.figure(figsize=(9, 5))
counts_series = full_aspects_per_rev.value_counts().sort_index()
sns.barplot(x=counts_series.index, y=counts_series.values, palette='mako', hue=counts_series.index, legend=False)
plt.title("4.10 Phân bố số lượng khía cạnh được đề cập trên mỗi bài đánh giá", fontweight='bold')
plt.xlabel("Số khía cạnh phân biệt (Aspect Count)")
plt.ylabel("Số lượng bài đánh giá")
for i, v in enumerate(counts_series.values):
    pct = (v / len(df_reviews)) * 100
    plt.text(i, v + 80, f"{v:,}\n({pct:.1f}%)", ha='center', fontsize=9)
plt.ylim(0, max(counts_series.values) * 1.15)
plt.tight_layout()
chart_path_410 = "data/chart_img/410_aspects_per_review_distribution.png"
plt.savefig(chart_path_410)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_410}'")

Đã lưu biểu đồ: 'data/chart_img/410_aspects_per_review_distribution.png'


### 4.11 Sơ đồ Quy trình Nghiên cứu (Research Pipeline)


In [22]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Polygon, Arc

os.makedirs("data/chart_img", exist_ok=True)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.family"] = "sans-serif"

def draw_rounded_card(ax, x, y, w, h, bg="#FFFFFF", edgecolor="none", lw=1.0, radius=0.24, shadow=True, zorder=3):
    if shadow:
        sh = FancyBboxPatch(
            (x + 0.03, y - 0.05), w, h,
            boxstyle=f"round,pad=0.0,rounding_size={radius}",
            facecolor="#000000", alpha=0.055, edgecolor="none", zorder=zorder - 1
        )
        ax.add_patch(sh)
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.0,rounding_size={radius}",
        facecolor=bg, edgecolor=edgecolor, linewidth=lw, zorder=zorder
    )
    ax.add_patch(box)

def draw_icon_data_layers(ax, cx, cy, scale=1.0, zorder=6):
    color = "#2997FF"
    w = 0.34 * scale
    h = 0.17 * scale
    for idx, dy in enumerate([-0.14 * scale, 0.0, 0.14 * scale]):
        y0 = cy + dy
        pts = np.array([
            [cx, y0 + h],
            [cx + w, y0],
            [cx, y0 - h],
            [cx - w, y0]
        ])
        if idx < 2:
            pts_halo = np.array([
                [cx, y0 + h + 0.045 * scale],
                [cx + w + 0.045 * scale, y0],
                [cx, y0 - h - 0.045 * scale],
                [cx - w - 0.045 * scale, y0]
            ])
            ax.add_patch(Polygon(pts_halo, closed=True, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + idx*2))
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="#FFFFFF", linewidth=1.5, zorder=zorder + idx*2 + 1))

def draw_icon_preprocessing_doc(ax, cx, cy, scale=1.0, zorder=6):
    color = "#8E6CEF"
    dw, dh = 0.44 * scale, 0.56 * scale
    x0, y0 = cx - dw / 2 - 0.03 * scale, cy - dh / 2
    doc = FancyBboxPatch(
        (x0, y0), dw, dh,
        boxstyle=f"round,pad=0.0,rounding_size={0.07*scale}",
        facecolor="none", edgecolor=color, linewidth=2.8, zorder=zorder
    )
    ax.add_patch(doc)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.10*scale], [y0 + dh - 0.16*scale, y0 + dh - 0.16*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.16*scale], [y0 + dh - 0.28*scale, y0 + dh - 0.28*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    mcx, mcy = x0 + dw - 0.05*scale, y0 + 0.11*scale
    r = 0.125 * scale
    ax.add_patch(Circle((mcx, mcy), r + 0.04*scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder+2))
    ax.add_patch(Circle((mcx, mcy), r, facecolor="#FFFFFF", edgecolor=color, linewidth=2.8, zorder=zorder+3))
    ax.plot([mcx + r*0.7, mcx + r*1.55], [mcy - r*0.7, mcy - r*1.55],
            color=color, lw=3.0, solid_capstyle="round", zorder=zorder+3)

def draw_icon_circular_arrow(ax, cx, cy, scale=1.0, zorder=6):
    color = "#F5428D"
    r = 0.27 * scale
    arc = Arc((cx, cy), 2*r, 2*r, angle=0, theta1=75, theta2=395, color=color, lw=3.0, capstyle="round", zorder=zorder)
    ax.add_patch(arc)
    ang = math.radians(68)
    ax_pt = cx + r * math.cos(ang)
    ay_pt = cy + r * math.sin(ang)
    head = np.array([
        [ax_pt + 0.08*scale, ay_pt + 0.01*scale],
        [ax_pt - 0.07*scale, ay_pt + 0.09*scale],
        [ax_pt - 0.04*scale, ay_pt - 0.08*scale]
    ])
    ax.add_patch(Polygon(head, closed=True, facecolor=color, edgecolor=color, lw=1.0, zorder=zorder+1))

def draw_icon_chart_nodes(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF5733"
    w, h = 0.52 * scale, 0.44 * scale
    x0, y0 = cx - w/2, cy - h/2
    ax.plot([x0, x0, x0 + w], [y0 + h, y0, y0], color=color, lw=2.8, solid_capstyle="round", zorder=zorder)
    pts = [
        (x0 + 0.12*scale, y0 + 0.16*scale),
        (x0 + 0.28*scale, y0 + 0.30*scale),
        (x0 + 0.45*scale, y0 + 0.21*scale)
    ]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=2.6, zorder=zorder+1)
    for px, py in pts:
        ax.add_patch(Circle((px, py), 0.048*scale, facecolor="#FFFFFF", edgecolor=color, lw=2.5, zorder=zorder+2))

def draw_icon_optimization(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF9500"
    bw, bh = 0.54 * scale, 0.11 * scale
    bar = FancyBboxPatch(
        (cx - bw/2, cy - bh/2), bw, bh,
        boxstyle=f"round,pad=0.0,rounding_size={0.055*scale}",
        facecolor="none", edgecolor=color, linewidth=2.6, zorder=zorder
    )
    ax.add_patch(bar)
    ax.plot([cx, cx], [cy + 0.31*scale, cy + 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy + 0.20*scale, cy + 0.10*scale, cy + 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx, cx], [cy - 0.31*scale, cy - 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy - 0.20*scale, cy - 0.10*scale, cy - 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)

def draw_icon_neural_network(ax, cx, cy, scale=1.0, zorder=6):
    cols_x = [cx - 0.36*scale, cx - 0.12*scale, cx + 0.12*scale, cx + 0.36*scale]
    cols_y = [
        [cy - 0.14*scale, cy + 0.14*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.14*scale, cy + 0.14*scale]
    ]
    col_colors = ["#8CE038", "#4CC764", "#1DB096", "#0099C6"]
    for c_i in range(3):
        c_mid = col_colors[c_i + 1]
        for y1 in cols_y[c_i]:
            for y2 in cols_y[c_i + 1]:
                ax.plot([cols_x[c_i], cols_x[c_i+1]], [y1, y2], color=c_mid, lw=1.7, alpha=0.85, zorder=zorder)
    for c_i in [1, 2]:
        ax.plot([cols_x[c_i], cols_x[c_i]], [cols_y[c_i][0], cols_y[c_i][-1]], color=col_colors[c_i], lw=1.4, alpha=0.6, zorder=zorder)
    for c_i in range(4):
        for y_n in cols_y[c_i]:
            ax.add_patch(Circle((cols_x[c_i], y_n), 0.052*scale, facecolor="#FFFFFF", edgecolor=col_colors[c_i], lw=2.3, zorder=zorder+2))

def build_figure_411():
    fig, ax = plt.subplots(figsize=(16.5, 5.5), dpi=300)
    ax.set_xlim(0, 16.5)
    ax.set_ylim(0, 5.5)
    ax.axis("off")

    draw_rounded_card(ax, 0.10, 0.10, 16.30, 5.30, bg="#F2F2F6", edgecolor="#E2E2E8", lw=1.5, radius=0.32, shadow=False, zorder=1)

    card_w = 2.08
    card_h = 1.92
    card_y = 1.38
    gap = 0.22
    start_x = 0.50

    steps = [
        ("Data", "9,990 Reviews", draw_icon_data_layers),
        ("Preprocessing", "1.pdf & NLTK", draw_icon_preprocessing_doc),
        ("Feature & EBR", "20 Feats | Parquet", draw_icon_circular_arrow),
        ("Multi-Task", "3-Class Sentiment", draw_icon_chart_nodes),
        ("Optimization", "Overlap & FP16", draw_icon_optimization),
    ]

    for i, (title, subtitle, icon_fn) in enumerate(steps):
        cx_box = start_x + i * (card_w + gap)
        draw_rounded_card(ax, cx_box, card_y, card_w, card_h, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
        center_x = cx_box + card_w / 2
        icon_fn(ax, center_x, card_y + 1.26, scale=0.95, zorder=6)
        ax.text(center_x, card_y + 0.57, title, ha="center", va="center",
                fontsize=14.0, fontweight="bold", color="#1D1D1F", zorder=7)
        ax.text(center_x, card_y + 0.24, subtitle, ha="center", va="center",
                fontsize=11.2, fontweight="normal", color="#515154", zorder=7)

        if i < len(steps) - 1:
            x1 = cx_box + card_w
            x2 = cx_box + card_w + gap
            ym = card_y + card_h / 2
            ax.plot([x1, x2], [ym, ym], color="#86868B", lw=2.0, zorder=2)

    last_card_right = start_x + 4 * (card_w + gap) + card_w
    right_box_x = 12.12
    right_box_y = 1.38
    right_box_w = 3.85
    right_box_h = 3.72
    ym_arrow = card_y + card_h / 2
    ax.plot([last_card_right, right_box_x - 0.04], [ym_arrow, ym_arrow], color="#86868B", lw=2.0, zorder=2)
    ax.plot([right_box_x - 0.16, right_box_x - 0.03, right_box_x - 0.16],
            [ym_arrow + 0.09, ym_arrow, ym_arrow - 0.09], color="#86868B", lw=2.0, solid_capstyle="round", zorder=2)

    draw_rounded_card(ax, right_box_x, right_box_y, right_box_w, right_box_h,
                      bg="#F5F5F7", edgecolor="#FFFFFF", lw=2.2, radius=0.26, shadow=True, zorder=3)
    rb_border = FancyBboxPatch(
        (right_box_x, right_box_y), right_box_w, right_box_h,
        boxstyle="round,pad=0.0,rounding_size=0.26",
        facecolor="none", edgecolor="#D8D8DE", linewidth=1.2, zorder=4
    )
    ax.add_patch(rb_border)

    rcx = right_box_x + right_box_w / 2
    ax.text(rcx, right_box_y + right_box_h - 0.34, "6-Aspect Heads", ha="center", va="center",
            fontsize=15.5, fontweight="bold", color="#1D1D1F", zorder=7)

    aspect_names = ["Service", "Facility", "Exper.", "Amenity", "Loyalty", "Brand."]
    bar_w = 0.44
    bar_h = 1.24
    bar_gap = 0.11
    total_bars_w = 6 * bar_w + 5 * bar_gap
    bar_start_x = rcx - total_bars_w / 2
    bar_y = right_box_y + 1.88
    bar_centers = []

    for a_i, a_lbl in enumerate(aspect_names):
        bx = bar_start_x + a_i * (bar_w + bar_gap)
        bcx = bx + bar_w / 2
        bar_centers.append(bcx)
        pill = FancyBboxPatch(
            (bx, bar_y), bar_w, bar_h,
            boxstyle="round,pad=0.0,rounding_size=0.08",
            facecolor="#0099C6", edgecolor="none", zorder=5
        )
        ax.add_patch(pill)
        for d_i in range(3):
            ax.add_patch(Circle((bcx, bar_y + bar_h - 0.12 - d_i * 0.065), 0.016, facecolor="#FFFFFF", edgecolor="none", zorder=6))
        ax.text(bcx, bar_y + 0.45, a_lbl, ha="center", va="center", rotation=90,
                fontsize=9.2, fontweight="bold", color="#FFFFFF", zorder=7)

    y_tree_top = bar_y
    y_tree_horiz = bar_y - 0.16
    y_tree_bot = bar_y - 0.38
    for bcx in bar_centers:
        ax.plot([bcx, bcx], [y_tree_top, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([bar_centers[0], bar_centers[-1]], [y_tree_horiz, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([rcx, rcx], [y_tree_horiz, y_tree_bot], color="#0099C6", lw=1.9, zorder=5)

    draw_icon_neural_network(ax, rcx, right_box_y + 1.04, scale=0.95, zorder=6)

    ax.text(rcx, right_box_y + 0.45, "CRAM-ABSA Model", ha="center", va="center",
            fontsize=15.5, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(rcx, right_box_y + 0.18, "3-Class Sentiment (Pos/Neu/Neg)", ha="center", va="center",
            fontsize=10.5, fontweight="normal", color="#515154", zorder=7)

    pill_x = start_x
    pill_w = (right_box_x + right_box_w) - start_x
    pill_y = 0.42
    pill_h = 0.64
    draw_rounded_card(ax, pill_x, pill_y, pill_w, pill_h, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.31, shadow=False, zorder=3)
    pcx = pill_x + pill_w / 2
    pcy = pill_y + pill_h / 2
    split_x = pcx + 0.85
    ax.text(split_x - 0.08, pcy, "Dual-Stream NLTK & EBR Grounding Principles",
            ha="right", va="center", fontsize=14.2, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(split_x + 0.08, pcy, "inform all stages of the pipeline",
            ha="left", va="center", fontsize=14.2, fontweight="normal", color="#1D1D1F", zorder=7)

    plt.savefig("data/chart_img/411_research_pipeline.png", bbox_inches="tight", pad_inches=0.04, facecolor="none")
    plt.close(fig)
    print("Generated refined 411_research_pipeline.png successfully.")

build_figure_411()

Generated refined 411_research_pipeline.png successfully.


### 4.12 Kiến trúc Tổng thể Đề xuất (Abstract Architecture of Proposed Method)


In [23]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Polygon, Arc

os.makedirs("data/chart_img", exist_ok=True)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.family"] = "sans-serif"

def draw_rounded_card(ax, x, y, w, h, bg="#FFFFFF", edgecolor="none", lw=1.0, radius=0.24, shadow=True, zorder=3):
    if shadow:
        sh = FancyBboxPatch(
            (x + 0.03, y - 0.05), w, h,
            boxstyle=f"round,pad=0.0,rounding_size={radius}",
            facecolor="#000000", alpha=0.055, edgecolor="none", zorder=zorder - 1
        )
        ax.add_patch(sh)
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.0,rounding_size={radius}",
        facecolor=bg, edgecolor=edgecolor, linewidth=lw, zorder=zorder
    )
    ax.add_patch(box)

def draw_step_arrow(ax, x1, x2, y, color="#86868B", lw=2.2, head_w=0.12, head_h=0.085, zorder=4):
    ax.plot([x1, x2 - 0.03], [y, y], color=color, lw=lw, solid_capstyle="round", zorder=zorder)
    ax.plot([x2 - head_w, x2 - 0.02, x2 - head_w],
            [y + head_h, y, y - head_h], color=color, lw=lw, solid_capstyle="round", zorder=zorder)

def draw_icon_data_layers(ax, cx, cy, scale=1.0, zorder=6):
    color = "#2997FF"
    w = 0.34 * scale
    h = 0.17 * scale
    for idx, dy in enumerate([-0.14 * scale, 0.0, 0.14 * scale]):
        y0 = cy + dy
        pts = np.array([
            [cx, y0 + h],
            [cx + w, y0],
            [cx, y0 - h],
            [cx - w, y0]
        ])
        if idx < 2:
            pts_halo = np.array([
                [cx, y0 + h + 0.045 * scale],
                [cx + w + 0.045 * scale, y0],
                [cx, y0 - h - 0.045 * scale],
                [cx - w - 0.045 * scale, y0]
            ])
            ax.add_patch(Polygon(pts_halo, closed=True, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + idx*2))
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="#FFFFFF", linewidth=1.5, zorder=zorder + idx*2 + 1))

def draw_icon_preprocessing_doc(ax, cx, cy, scale=1.0, zorder=6):
    color = "#8E6CEF"
    dw, dh = 0.44 * scale, 0.56 * scale
    x0, y0 = cx - dw / 2 - 0.03 * scale, cy - dh / 2
    doc = FancyBboxPatch(
        (x0, y0), dw, dh,
        boxstyle=f"round,pad=0.0,rounding_size={0.07*scale}",
        facecolor="none", edgecolor=color, linewidth=2.8, zorder=zorder
    )
    ax.add_patch(doc)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.10*scale], [y0 + dh - 0.16*scale, y0 + dh - 0.16*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.16*scale], [y0 + dh - 0.28*scale, y0 + dh - 0.28*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    mcx, mcy = x0 + dw - 0.05*scale, y0 + 0.11*scale
    r = 0.125 * scale
    ax.add_patch(Circle((mcx, mcy), r + 0.04*scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder+2))
    ax.add_patch(Circle((mcx, mcy), r, facecolor="#FFFFFF", edgecolor=color, linewidth=2.8, zorder=zorder+3))
    ax.plot([mcx + r*0.7, mcx + r*1.55], [mcy - r*0.7, mcy - r*1.55],
            color=color, lw=3.0, solid_capstyle="round", zorder=zorder+3)

def draw_icon_circular_arrow(ax, cx, cy, scale=1.0, zorder=6):
    color = "#F5428D"
    r = 0.27 * scale
    arc = Arc((cx, cy), 2*r, 2*r, angle=0, theta1=75, theta2=395, color=color, lw=3.0, capstyle="round", zorder=zorder)
    ax.add_patch(arc)
    ang = math.radians(68)
    ax_pt = cx + r * math.cos(ang)
    ay_pt = cy + r * math.sin(ang)
    head = np.array([
        [ax_pt + 0.08*scale, ay_pt + 0.01*scale],
        [ax_pt - 0.07*scale, ay_pt + 0.09*scale],
        [ax_pt - 0.04*scale, ay_pt - 0.08*scale]
    ])
    ax.add_patch(Polygon(head, closed=True, facecolor=color, edgecolor=color, lw=1.0, zorder=zorder+1))

def draw_icon_chart_nodes(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF5733"
    w, h = 0.52 * scale, 0.44 * scale
    x0, y0 = cx - w/2, cy - h/2
    ax.plot([x0, x0, x0 + w], [y0 + h, y0, y0], color=color, lw=2.8, solid_capstyle="round", zorder=zorder)
    pts = [
        (x0 + 0.12*scale, y0 + 0.16*scale),
        (x0 + 0.28*scale, y0 + 0.30*scale),
        (x0 + 0.45*scale, y0 + 0.21*scale)
    ]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=2.6, zorder=zorder+1)
    for px, py in pts:
        ax.add_patch(Circle((px, py), 0.048*scale, facecolor="#FFFFFF", edgecolor=color, lw=2.5, zorder=zorder+2))

def draw_icon_optimization(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF9500"
    bw, bh = 0.54 * scale, 0.11 * scale
    bar = FancyBboxPatch(
        (cx - bw/2, cy - bh/2), bw, bh,
        boxstyle=f"round,pad=0.0,rounding_size={0.055*scale}",
        facecolor="none", edgecolor=color, linewidth=2.6, zorder=zorder
    )
    ax.add_patch(bar)
    ax.plot([cx, cx], [cy + 0.31*scale, cy + 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy + 0.20*scale, cy + 0.10*scale, cy + 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx, cx], [cy - 0.31*scale, cy - 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy - 0.20*scale, cy - 0.10*scale, cy - 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)

def draw_icon_neural_network(ax, cx, cy, scale=1.0, zorder=6):
    cols_x = [cx - 0.36*scale, cx - 0.12*scale, cx + 0.12*scale, cx + 0.36*scale]
    cols_y = [
        [cy - 0.14*scale, cy + 0.14*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.14*scale, cy + 0.14*scale]
    ]
    col_colors = ["#8CE038", "#4CC764", "#1DB096", "#0099C6"]
    for c_i in range(3):
        c_mid = col_colors[c_i + 1]
        for y1 in cols_y[c_i]:
            for y2 in cols_y[c_i + 1]:
                ax.plot([cols_x[c_i], cols_x[c_i+1]], [y1, y2], color=c_mid, lw=1.7, alpha=0.85, zorder=zorder)
    for c_i in [1, 2]:
        ax.plot([cols_x[c_i], cols_x[c_i]], [cols_y[c_i][0], cols_y[c_i][-1]], color=col_colors[c_i], lw=1.4, alpha=0.6, zorder=zorder)
    for c_i in range(4):
        for y_n in cols_y[c_i]:
            ax.add_patch(Circle((cols_x[c_i], y_n), 0.052*scale, facecolor="#FFFFFF", edgecolor=col_colors[c_i], lw=2.3, zorder=zorder+2))

def build_figure_412():
    fig, ax = plt.subplots(figsize=(21.8, 4.9), dpi=300)
    ax.set_xlim(0, 21.8)
    ax.set_ylim(0, 4.9)
    ax.axis("off")

    draw_rounded_card(ax, 0.10, 0.10, 21.60, 4.70, bg="#F2F2F6", edgecolor="#E2E2E8", lw=1.5, radius=0.32, shadow=False, zorder=1)

    steps = [
        ("Dual-Stream Input", "Text Cleaning + NLTK", draw_icon_preprocessing_doc),
        ("20-Feature & EBR", "Parquet Store | 20 Feats", draw_icon_data_layers),
        ("BERT Backbone", "bert-base-uncased (768d)", draw_icon_neural_network),
        ("Feature Fusion", "Residual Gate (20 -> 768d)", draw_icon_circular_arrow),
        ("Rationale Masking", "Soft Mask m_a | Overlap Loss", draw_icon_optimization),
        ("3-Class Sentiment", "Pos/Neu/Neg | Acc@0 & Macro-MAE", draw_icon_chart_nodes),
    ]

    start_x = 0.45
    card_w = 2.52
    card_h = 2.02
    gap = 0.34
    right_box_y = 0.42
    right_box_h = 4.06
    card_y = right_box_y + (right_box_h - card_h) / 2

    for i, (title, subtitle, icon_fn) in enumerate(steps):
        cx_box = start_x + i * (card_w + gap)
        draw_rounded_card(ax, cx_box, card_y, card_w, card_h, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
        center_x = cx_box + card_w / 2
        icon_fn(ax, center_x, card_y + 1.34, scale=0.92, zorder=6)
        ax.text(center_x, card_y + 0.62, title, ha="center", va="center",
                fontsize=12.4, fontweight="bold", color="#1D1D1F", zorder=7)
        ax.text(center_x, card_y + 0.26, subtitle, ha="center", va="center",
                fontsize=9.2, fontweight="normal", color="#515154", zorder=7)

        if i < len(steps) - 1:
            x1 = cx_box + card_w + 0.03
            x2 = cx_box + card_w + gap - 0.03
            ym = card_y + card_h / 2
            draw_step_arrow(ax, x1, x2, ym, color="#86868B", lw=2.1, head_w=0.11, head_h=0.08, zorder=4)

    last_card_right = start_x + (len(steps) - 1) * (card_w + gap) + card_w
    right_box_x = last_card_right + gap
    right_box_w = 3.82

    ym_arrow = card_y + card_h / 2
    draw_step_arrow(ax, last_card_right + 0.03, right_box_x - 0.03, ym_arrow, color="#86868B", lw=2.1, head_w=0.11, head_h=0.08, zorder=4)

    draw_rounded_card(ax, right_box_x, right_box_y, right_box_w, right_box_h,
                      bg="#F5F5F7", edgecolor="#FFFFFF", lw=2.2, radius=0.26, shadow=True, zorder=3)
    rb_border = FancyBboxPatch(
        (right_box_x, right_box_y), right_box_w, right_box_h,
        boxstyle="round,pad=0.0,rounding_size=0.26",
        facecolor="none", edgecolor="#D8D8DE", linewidth=1.2, zorder=4
    )
    ax.add_patch(rb_border)

    rcx = right_box_x + right_box_w / 2
    ax.text(rcx, right_box_y + right_box_h - 0.35, "6-Aspect Adapters", ha="center", va="center",
            fontsize=15.2, fontweight="bold", color="#1D1D1F", zorder=7)

    aspect_names = ["Service", "Facility", "Exper.", "Amenity", "Loyalty", "Brand."]
    bar_w = 0.43
    bar_h = 1.38
    bar_gap = 0.105
    total_bars_w = 6 * bar_w + 5 * bar_gap
    bar_start_x = rcx - total_bars_w / 2
    bar_y = right_box_y + 2.04
    bar_centers = []

    for a_i, a_lbl in enumerate(aspect_names):
        bx = bar_start_x + a_i * (bar_w + bar_gap)
        bcx = bx + bar_w / 2
        bar_centers.append(bcx)
        pill = FancyBboxPatch(
            (bx, bar_y), bar_w, bar_h,
            boxstyle="round,pad=0.0,rounding_size=0.08",
            facecolor="#0099C6", edgecolor="none", zorder=5
        )
        ax.add_patch(pill)
        for d_i in range(3):
            ax.add_patch(Circle((bcx, bar_y + bar_h - 0.13 - d_i * 0.068), 0.016, facecolor="#FFFFFF", edgecolor="none", zorder=6))
        ax.text(bcx, bar_y + 0.50, a_lbl, ha="center", va="center", rotation=90,
                fontsize=9.4, fontweight="bold", color="#FFFFFF", zorder=7)

    y_tree_top = bar_y
    y_tree_horiz = bar_y - 0.17
    y_tree_bot = bar_y - 0.40
    for bcx in bar_centers:
        ax.plot([bcx, bcx], [y_tree_top, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([bar_centers[0], bar_centers[-1]], [y_tree_horiz, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([rcx, rcx], [y_tree_horiz, y_tree_bot + 0.03], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([rcx - 0.065, rcx, rcx + 0.065], [y_tree_bot + 0.09, y_tree_bot + 0.01, y_tree_bot + 0.09],
            color="#0099C6", lw=1.9, solid_capstyle="round", zorder=5)

    out_card_w = right_box_w - 0.42
    out_card_h = 1.38
    out_card_x = rcx - out_card_w / 2
    out_card_y = right_box_y + 0.22
    draw_rounded_card(ax, out_card_x, out_card_y, out_card_w, out_card_h,
                      bg="#FFFFFF", edgecolor="#DCE6F2", lw=1.3, radius=0.18, shadow=False, zorder=5)
    ax.text(rcx, out_card_y + 1.04, "CRAM-ABSA Output", ha="center", va="center",
            fontsize=14.4, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(rcx, out_card_y + 0.65, "Aspect: Mentioned List", ha="center", va="center",
            fontsize=11.4, fontweight="bold", color="#0066CC", zorder=7)
    ax.text(rcx, out_card_y + 0.28, "3-Class Sentiment (Pos/Neu/Neg)", ha="center", va="center",
            fontsize=10.8, fontweight="normal", color="#333336", zorder=7)

    plt.savefig("data/chart_img/412_abstract_architecture_of_proposed_method.png", bbox_inches="tight", pad_inches=0.04, facecolor="none")
    plt.close(fig)
    print("Generated refined 412_abstract_architecture_of_proposed_method.png successfully.")

build_figure_412()

Generated refined 412_abstract_architecture_of_proposed_method.png successfully.


## 5. Feature Engineering


### 5.1 Tạo 20 Features


In [24]:
try:
    from nltk.corpus import stopwords
    NLTK_STOPWORDS = set(stopwords.words("english"))
except Exception:
    NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once"
    }

CONTRACTIONS_MAP = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing", "nowhere", "hardly",
    "barely", "scarcely", "without", "but", "however", "although", "though", "yet",
    "despite", "unfortunately", "except", "fault"
}

HEDGING_MODIFIERS = {
    "okay", "ok", "decent", "average", "fine", "fair", "moderate", "standard",
    "acceptable", "passable", "normal", "typical", "ordinary", "somewhat", "slightly",
    "maybe", "perhaps", "quite", "rather", "relatively", "adequate", "mediocre",
    "reasonable", "reasonably"
}

SAFE_STOPWORDS = NLTK_STOPWORDS - NEGATION_AND_CONTRAST_WORDS

OPINION_LEXICON_POS = {
    "good", "great", "excellent", "amazing", "wonderful", "friendly", "helpful", "clean",
    "spacious", "comfortable", "delicious", "lovely", "perfect", "best", "nice", "fantastic",
    "superb", "polite", "attentive", "convenient", "beautiful", "cozy", "relaxing", "awesome",
    "recommend", "recommended", "happy", "pleased", "impressed", "welcoming", "professional",
    "quiet", "modern", "fresh", "tasty", "worth", "luxury", "luxurious", "outstanding", "warm"
}

OPINION_LEXICON_NEG = {
    "bad", "poor", "terrible", "horrible", "awful", "slow", "rude", "dirty", "noisy", "small",
    "old", "outdated", "broken", "cold", "hard", "uncomfortable", "overpriced", "expensive",
    "worst", "disappointed", "disappointing", "unfriendly", "unhelpful", "smell", "smelly",
    "stain", "stained", "cockroach", "bug", "mold", "issue", "problem", "complaint", "wait",
    "waiting", "delayed", "mediocre", "basic", "boring", "crowded", "weak", "fault"
}

INTENSIFIERS = {
    "very", "extremely", "really", "super", "highly", "totally", "absolutely", "completely",
    "quite", "pretty", "so", "too", "exceptionally", "incredibly", "amazingly", "remarkably"
}

CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0
SENTIMENT_VAL_MAP = {"Positive": 1.0, "Neutral": 0.0, "Negative": -1.0}

ASPECT_KEYWORDS = {
    "Service": {"staff", "service", "reception", "front", "desk", "manager", "waiter", "waitress", "concierge", "bellboy", "helpful", "friendly", "polite", "check", "checkin", "checkout", "attendant", "housekeeping", "host", "crew"},
    "Facility": {"room", "bed", "bathroom", "shower", "toilet", "air", "ac", "conditioning", "clean", "cleanliness", "noise", "noisy", "soundproof", "view", "balcony", "floor", "elevator", "lift", "pillow", "mattress", "furniture", "decor", "water", "carpet"},
    "Experience": {"stay", "stayed", "experience", "holiday", "vacation", "trip", "night", "overall", "atmosphere", "vibe", "relax", "relaxing", "enjoy", "enjoyed", "memorable", "time", "comfort", "comfortable", "quiet", "peaceful", "feeling"},
    "Amenity": {"breakfast", "buffet", "pool", "swimming", "spa", "gym", "fitness", "wifi", "internet", "restaurant", "bar", "food", "drink", "coffee", "tea", "lounge", "shuttle", "parking", "massage", "rooftop"},
    "Loyalty": {"back", "return", "again", "recommend", "recommended", "definitely", "choose", "future", "next", "visit", "repeat", "choice", "staying"},
    "Branding": {"star", "stars", "luxury", "brand", "hotel", "resort", "standard", "reputation", "class", "five", "four", "chain", "management", "value", "price", "worth", "quality", "prestigious", "accor", "sofitel", "pullman", "novotel", "marriott", "hilton", "intercontinental", "sheraton"}
}

def nltk_negation_aware_preprocess(text: str):
    lower_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        lower_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, lower_txt)
        lower_txt = lower_txt.replace(k, v)
    lower_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", lower_txt)
    raw_tokens = re.findall(r"[a-z]+", lower_txt)
    clean_tokens = [t for t in raw_tokens if t not in SAFE_STOPWORDS and len(t) > 1]
    return raw_tokens, clean_tokens

def preprocess_text_paper1(text: str):
    norm_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        norm_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, norm_txt)
        norm_txt = norm_txt.replace(k, v)
    norm_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", norm_txt)
    clean_review_text = re.sub(r"[^a-z0-9.,!?'\s-]", " ", norm_txt)
    clean_review_text = re.sub(r"\s+", " ", clean_review_text).strip()
    _, clean_tokens = nltk_negation_aware_preprocess(norm_txt)
    return clean_review_text, " ".join(clean_tokens)

def infer_aspect_sentiment_from_text_nltk(raw_text: str, target_asp: str) -> float:
    sentences = [s.strip() for s in re.split(r"[.!?]+", str(raw_text)) if s.strip()]
    kw_set = ASPECT_KEYWORDS[target_asp]
    scores = []
    for sent in sentences:
        raw_toks, clean_toks = nltk_negation_aware_preprocess(sent)
        if any(t in kw_set for t in clean_toks):
            pos_hits = sum(1 for t in raw_toks if t in OPINION_LEXICON_POS)
            neg_hits = sum(1 for t in raw_toks if t in OPINION_LEXICON_NEG)
            for idx_t, tok in enumerate(raw_toks[:-1]):
                if tok in {"not", "never", "no", "hardly"}:
                    nxt = raw_toks[idx_t + 1]
                    if nxt in OPINION_LEXICON_POS:
                        pos_hits = max(0, pos_hits - 1)
                        neg_hits += 1
                    elif nxt in OPINION_LEXICON_NEG:
                        neg_hits = max(0, neg_hits - 1)
                        pos_hits += 1
            scores.append((pos_hits - neg_hits) / (pos_hits + neg_hits) if (pos_hits + neg_hits) > 0 else 0.0)
    return float(round(sum(scores) / len(scores), 4)) if scores else 0.0

df_features = pd.DataFrame({"review_id": df_reviews["review_id"]})
ann_by_review = defaultdict(list)
for rec in df_annotations[["review_id", "aspect", "sentiment", "start_char", "end_char"]].to_dict("records"):
    ann_by_review[int(rec["review_id"])].append(rec)

review_cache = {}
for row in df_reviews[["review_id", "review_text", "char_length"]].to_dict("records"):
    rid = int(row["review_id"])
    raw_txt = str(row["review_text"])
    raw_toks, clean_toks = nltk_negation_aware_preprocess(raw_txt)
    recs = ann_by_review.get(rid, [])
    inferred = {asp: infer_aspect_sentiment_from_text_nltk(raw_txt, asp) for asp in CANONICAL_ASPECTS} if not recs else {}
    review_cache[rid] = {
        "raw_text": raw_txt,
        "char_len": int(row["char_length"]),
        "raw_tokens": raw_toks,
        "clean_tokens": clean_toks,
        "records": recs,
        "inferred": inferred
    }

def compute_aspect_polarity(rid: int, target_asp: str) -> float:
    c = review_cache[rid]
    recs = [r for r in c["records"] if r["aspect"] == target_asp]
    if not recs:
        return c["inferred"].get(target_asp, 0.0) if not c["records"] else 0.0
    w_sum = sum(max(1, int(r["end_char"]) - int(r["start_char"])) * SENTIMENT_VAL_MAP.get(r["sentiment"], 0.0) for r in recs)
    t_len = sum(max(1, int(r["end_char"]) - int(r["start_char"])) for r in recs)
    return float(round(w_sum / t_len, 4)) if t_len > 0 else 0.0

print(f"Đã khởi tạo df_features và bộ nhớ đệm tiền xử lý cho {len(df_features):,} reviews.")

Đã khởi tạo df_features và bộ nhớ đệm tiền xử lý cho 9,990 reviews.


#### Feature 01: `feat_aspect_count`
Đếm số lượng khía cạnh phân biệt xuất hiện trong bài đánh giá (từ 0 đến 6). Đặc trưng này phản ánh độ rộng thông tin để mô hình ABSA điều chỉnh trọng số chú ý đa khía cạnh.


In [25]:
def calc_feat_aspect_count(rid):
    c = review_cache[rid]
    if c["records"]:
        return len({r["aspect"] for r in c["records"]})
    return sum(1 for asp, sc in c["inferred"].items() if abs(sc) > 0 or any(t in ASPECT_KEYWORDS[asp] for t in c["clean_tokens"]))

df_features["feat_aspect_count"] = [calc_feat_aspect_count(int(r)) for r in df_features["review_id"]]
print(f"feat_aspect_count -> mean: {df_features['feat_aspect_count'].mean():.4f}, min: {df_features['feat_aspect_count'].min()}, max: {df_features['feat_aspect_count'].max()}")

feat_aspect_count -> mean: 3.3973, min: 0, max: 6


#### Feature 02: `feat_dominant_aspect`
Mã hóa ID (1..6, hoặc 0 nếu không có) của khía cạnh có tần suất xuất hiện nhiều nhất trong bài đánh giá. Giúp định hướng khía cạnh trọng tâm quyết định nhãn cảm xúc 3 lớp (Negative, Neutral, Positive).


In [26]:
def calc_feat_dominant_aspect(rid):
    c = review_cache[rid]
    if c["records"]:
        counts = Counter(r["aspect"] for r in c["records"])
        dom = counts.most_common(1)[0][0]
        return int(ASPECT_ID_MAP.get(dom, 0))
    active = [asp for asp, sc in c["inferred"].items() if abs(sc) > 0 or any(t in ASPECT_KEYWORDS[asp] for t in c["clean_tokens"])]
    return int(ASPECT_ID_MAP.get(active[0], 0)) if active else 0

df_features["feat_dominant_aspect"] = [calc_feat_dominant_aspect(int(r)) for r in df_features["review_id"]]
print(f"feat_dominant_aspect -> mode: {int(df_features['feat_dominant_aspect'].mode().iloc[0])}, unique IDs: {sorted(df_features['feat_dominant_aspect'].unique().tolist())}")

feat_dominant_aspect -> mode: 1, unique IDs: [0, 1, 2, 3, 4, 5, 6]


#### Feature 03: `feat_net_sentiment_score`
Hiệu số chuẩn hóa giữa số lượng đoạn bằng chứng Positive và Negative chia cho tổng số đoạn bằng chứng trong `[-1.0, 1.0]`. Cung cấp tín hiệu phân cực tổng hợp trực tiếp cho bộ phân loại cảm xúc 3 lớp.


In [27]:
def calc_feat_net_sentiment_score(rid):
    c = review_cache[rid]
    recs = c["records"]
    if recs:
        pos_c = sum(1 for r in recs if r["sentiment"] == "Positive")
        neg_c = sum(1 for r in recs if r["sentiment"] == "Negative")
        return float(round((pos_c - neg_c) / len(recs), 4))
    vals = [v for v in c["inferred"].values() if abs(v) > 0]
    return float(round(sum(vals) / len(vals), 4)) if vals else 0.0

df_features["feat_net_sentiment_score"] = [calc_feat_net_sentiment_score(int(r)) for r in df_features["review_id"]]
print(f"feat_net_sentiment_score -> mean: {df_features['feat_net_sentiment_score'].mean():.4f}, min: {df_features['feat_net_sentiment_score'].min():.4f}, max: {df_features['feat_net_sentiment_score'].max():.4f}")

feat_net_sentiment_score -> mean: 0.7741, min: -1.0000, max: 1.0000


#### Feature 04: `feat_sentiment_conflict`
Cờ nhị phân `{0, 1}` bật lên 1 khi bài đánh giá vừa có khía cạnh Positive vừa có khía cạnh Negative. Hỗ trợ nhận diện các bài đánh giá đa chiều dễ rơi vào lớp Neutral hoặc có xung đột ngữ nghĩa.


In [28]:
def calc_feat_sentiment_conflict(rid):
    c = review_cache[rid]
    recs = c["records"]
    if recs:
        has_pos = any(r["sentiment"] == "Positive" for r in recs)
        has_neg = any(r["sentiment"] == "Negative" for r in recs)
        return 1 if (has_pos and has_neg) else 0
    vals = list(c["inferred"].values())
    return 1 if (any(v > 0 for v in vals) and any(v < 0 for v in vals)) else 0

df_features["feat_sentiment_conflict"] = [calc_feat_sentiment_conflict(int(r)) for r in df_features["review_id"]]
print(f"feat_sentiment_conflict -> conflict rate: {df_features['feat_sentiment_conflict'].mean()*100:.2f}% ({df_features['feat_sentiment_conflict'].sum():,} reviews)")

feat_sentiment_conflict -> conflict rate: 16.66% (1,664 reviews)


#### Feature 05: `feat_evidence_density`
Tỷ lệ tổng số ký tự thuộc các đoạn bằng chứng (rationale spans) trên tổng độ dài ký tự bài đánh giá trong `[0.0, 1.0]`. Đo lường mật độ thông tin hữu ích dùng làm trọng số tin cậy khi tổng hợp đa đánh giá.


In [29]:
def calc_feat_evidence_density(rid):
    c = review_cache[rid]
    if c["records"] and c["char_len"] > 0:
        span_chars = sum(max(0, int(r["end_char"]) - int(r["start_char"])) for r in c["records"])
        return float(round(min(span_chars / c["char_len"], 1.0), 4))
    return float(round(min(len(c["clean_tokens"]) / max(len(c["raw_tokens"]), 1), 1.0), 4))

df_features["feat_evidence_density"] = [calc_feat_evidence_density(int(r)) for r in df_features["review_id"]]
print(f"feat_evidence_density -> mean: {df_features['feat_evidence_density'].mean():.4f}, min: {df_features['feat_evidence_density'].min():.4f}, max: {df_features['feat_evidence_density'].max():.4f}")

feat_evidence_density -> mean: 0.7954, min: 0.0419, max: 1.0000


#### Feature 06: `feat_avg_rationale_len`
Độ dài ký tự trung bình của các đoạn bằng chứng trong bài đánh giá. Phản ánh mức độ chi tiết của lập luận khi khách hàng khen hoặc chê từng khía cạnh.


In [30]:
def calc_feat_avg_rationale_len(rid):
    c = review_cache[rid]
    if c["records"]:
        lengths = [max(0, int(r["end_char"]) - int(r["start_char"])) for r in c["records"]]
        return float(round(sum(lengths) / len(lengths), 2))
    return float(round(c["char_len"] / max(int(df_features.loc[rid - 1, "feat_aspect_count"]), 1), 2))

df_features["feat_avg_rationale_len"] = [calc_feat_avg_rationale_len(int(r)) for r in df_features["review_id"]]
print(f"feat_avg_rationale_len -> mean: {df_features['feat_avg_rationale_len'].mean():.2f}, min: {df_features['feat_avg_rationale_len'].min():.2f}, max: {df_features['feat_avg_rationale_len'].max():.2f}")

feat_avg_rationale_len -> mean: 66.63, min: 16.00, max: 394.50


#### Feature 07: `feat_has_cross_overlap`
Cờ nhị phân `{0, 1}` kiểm tra sự chồng lấn tọa độ ký tự giữa hai khía cạnh khác nhau trong cùng một bài đánh giá. Cung cấp tín hiệu điều phạt chồng lấn (Overlap Penalty) cho cơ chế Rationale Masking.


In [31]:
def calc_feat_has_cross_overlap(rid):
    recs = review_cache[rid]["records"]
    n = len(recs)
    for i in range(n):
        for j in range(i + 1, n):
            if recs[i]["aspect"] != recs[j]["aspect"]:
                if not (recs[i]["end_char"] <= recs[j]["start_char"] or recs[i]["start_char"] >= recs[j]["end_char"]):
                    return 1
    return 0

df_features["feat_has_cross_overlap"] = [calc_feat_has_cross_overlap(int(r)) for r in df_features["review_id"]]
print(f"feat_has_cross_overlap -> overlapping reviews: {df_features['feat_has_cross_overlap'].sum()} / {len(df_features):,}")

feat_has_cross_overlap -> overlapping reviews: 6 / 9,990


#### Feature 08: `feat_service_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Service` trong `[-1.0, 1.0]`. Biểu diễn trực tiếp thái độ của khách hàng đối với chất lượng phục vụ và nhân viên khách sạn.


In [32]:
df_features["feat_service_sentiment"] = [compute_aspect_polarity(int(r), "Service") for r in df_features["review_id"]]
print(f"feat_service_sentiment -> mean: {df_features['feat_service_sentiment'].mean():.4f}, non-zero: {(df_features['feat_service_sentiment'] != 0).sum():,}")

feat_service_sentiment -> mean: 0.7559, non-zero: 9,044


#### Feature 09: `feat_facility_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Facility` trong `[-1.0, 1.0]`. Phản ánh đánh giá về phòng ốc, giường ngủ, cách âm và trang thiết bị nội thất.


In [33]:
df_features["feat_facility_sentiment"] = [compute_aspect_polarity(int(r), "Facility") for r in df_features["review_id"]]
print(f"feat_facility_sentiment -> mean: {df_features['feat_facility_sentiment'].mean():.4f}, non-zero: {(df_features['feat_facility_sentiment'] != 0).sum():,}")

feat_facility_sentiment -> mean: 0.5358, non-zero: 7,200


#### Feature 10: `feat_experience_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Experience` trong `[-1.0, 1.0]`. Đo lường mức độ hài lòng tổng thể về kỳ nghỉ, không gian và sự thoải mái khi lưu trú.


In [34]:
df_features["feat_experience_sentiment"] = [compute_aspect_polarity(int(r), "Experience") for r in df_features["review_id"]]
print(f"feat_experience_sentiment -> mean: {df_features['feat_experience_sentiment'].mean():.4f}, non-zero: {(df_features['feat_experience_sentiment'] != 0).sum():,}")

feat_experience_sentiment -> mean: 0.4602, non-zero: 5,908


#### Feature 11: `feat_amenity_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Amenity` trong `[-1.0, 1.0]`. Ghi nhận cảm xúc 3 lớp về các tiện ích đi kèm như bữa sáng, hồ bơi, quầy bar, spa và wifi.


In [35]:
df_features["feat_amenity_sentiment"] = [compute_aspect_polarity(int(r), "Amenity") for r in df_features["review_id"]]
print(f"feat_amenity_sentiment -> mean: {df_features['feat_amenity_sentiment'].mean():.4f}, non-zero: {(df_features['feat_amenity_sentiment'] != 0).sum():,}")

feat_amenity_sentiment -> mean: 0.4850, non-zero: 5,457


#### Feature 12: `feat_loyalty_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Loyalty` trong `[-1.0, 1.0]`. Nắm bắt ý định quay lại hoặc sẵn sàng giới thiệu khách sạn của khách hàng.


In [36]:
df_features["feat_loyalty_sentiment"] = [compute_aspect_polarity(int(r), "Loyalty") for r in df_features["review_id"]]
print(f"feat_loyalty_sentiment -> mean: {df_features['feat_loyalty_sentiment'].mean():.4f}, non-zero: {(df_features['feat_loyalty_sentiment'] != 0).sum():,}")

feat_loyalty_sentiment -> mean: 0.3532, non-zero: 3,928


#### Feature 13: `feat_branding_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Branding` trong `[-1.0, 1.0]`. Đánh giá nhận thức về đẳng cấp thương hiệu, tiêu chuẩn khách sạn và tương xứng giá trị.


In [37]:
df_features["feat_branding_sentiment"] = [compute_aspect_polarity(int(r), "Branding") for r in df_features["review_id"]]
print(f"feat_branding_sentiment -> mean: {df_features['feat_branding_sentiment'].mean():.4f}, non-zero: {(df_features['feat_branding_sentiment'] != 0).sum():,}")

feat_branding_sentiment -> mean: 0.0805, non-zero: 1,233


#### Feature 14: `feat_aspect_coverage_ratio`
Tỷ lệ bao phủ khía cạnh tính bằng `min(feat_aspect_count / 6.0, 1.0)` trong `[0.0, 1.0]`. Chuẩn hóa số lượng khía cạnh được đề cập về thang đo đơn vị cho tầng Feature Fusion.


In [38]:
df_features["feat_aspect_coverage_ratio"] = (df_features["feat_aspect_count"] / 6.0).clip(0.0, 1.0).round(4)
print(f"feat_aspect_coverage_ratio -> mean: {df_features['feat_aspect_coverage_ratio'].mean():.4f}, min: {df_features['feat_aspect_coverage_ratio'].min():.4f}, max: {df_features['feat_aspect_coverage_ratio'].max():.4f}")

feat_aspect_coverage_ratio -> mean: 0.5662, min: 0.0000, max: 1.0000


#### Feature 15: `feat_neutral_hedging_density`
Mật độ từ ngữ trung tính, rào đón và tương phản (`HEDGING_MODIFIERS | {'but', 'however', 'although', 'though', 'yet', 'despite'}`) trên tổng số từ gốc trong `[0.0, 1.0]`. Đặc trưng then chốt giúp phân biệt lớp `Neutral` với hai cực `Positive` và `Negative`.


In [39]:
HEDGING_CONTRAST_SET = HEDGING_MODIFIERS | {"but", "however", "although", "though", "yet", "despite"}

def calc_feat_neutral_hedging_density(rid):
    raw_toks = review_cache[rid]["raw_tokens"]
    if not raw_toks:
        return 0.0
    hits = sum(1 for t in raw_toks if t in HEDGING_CONTRAST_SET)
    return float(round(min(hits / len(raw_toks), 1.0), 4))

df_features["feat_neutral_hedging_density"] = [calc_feat_neutral_hedging_density(int(r)) for r in df_features["review_id"]]
print(f"feat_neutral_hedging_density -> mean: {df_features['feat_neutral_hedging_density'].mean():.4f}, max: {df_features['feat_neutral_hedging_density'].max():.4f}")

feat_neutral_hedging_density -> mean: 0.0109, max: 0.1538


#### Feature 16: `feat_consensus_strength`
Độ mạnh đồng thuận cảm xúc nội tại tính bằng `abs(feat_net_sentiment_score)` trong `[0.0, 1.0]`. Giá trị gần 1 biểu thị sự nhất quán một chiều (Positive hoặc Negative), giá trị gần 0 báo hiệu trạng thái cân bằng/Neutral.


In [40]:
df_features["feat_consensus_strength"] = df_features["feat_net_sentiment_score"].abs().round(4)
print(f"feat_consensus_strength -> mean: {df_features['feat_consensus_strength'].mean():.4f}, min: {df_features['feat_consensus_strength'].min():.4f}, max: {df_features['feat_consensus_strength'].max():.4f}")

feat_consensus_strength -> mean: 0.8602, min: 0.0000, max: 1.0000


#### Feature 17: `feat_lexical_polarity_intensity`
Cường độ phân cực từ vựng có xét đảo chiều phủ định và khuếch đại trạng từ (`INTENSIFIERS`) trong `[-1.0, 1.0]`. Bổ sung tín hiệu cảm xúc mức từ vựng độc lập với nhãn chú thích thủ công.


In [41]:
def calc_feat_lexical_polarity_intensity(rid):
    raw_toks = review_cache[rid]["raw_tokens"]
    pos_cnt = sum(1 for t in raw_toks if t in OPINION_LEXICON_POS)
    neg_cnt = sum(1 for t in raw_toks if t in OPINION_LEXICON_NEG)
    intens_cnt = sum(1 for t in raw_toks if t in INTENSIFIERS)
    for idx_t, tok in enumerate(raw_toks[:-1]):
        if tok in {"not", "never", "no", "hardly", "neither", "nor", "without"}:
            nxt = raw_toks[idx_t + 1]
            if nxt in OPINION_LEXICON_POS:
                pos_cnt = max(0, pos_cnt - 1)
                neg_cnt += 1
            elif nxt in OPINION_LEXICON_NEG:
                neg_cnt = max(0, neg_cnt - 1)
                pos_cnt += 1
    base_pol = (pos_cnt - neg_cnt) / max(pos_cnt + neg_cnt, 1) if (pos_cnt + neg_cnt) > 0 else 0.0
    intens_boost = min(0.35, 0.12 * intens_cnt) * (1.0 if base_pol >= 0 else -1.0) if base_pol != 0 else 0.0
    return float(round(np.clip(base_pol + intens_boost, -1.0, 1.0), 4))

df_features["feat_lexical_polarity_intensity"] = [calc_feat_lexical_polarity_intensity(int(r)) for r in df_features["review_id"]]
print(f"feat_lexical_polarity_intensity -> mean: {df_features['feat_lexical_polarity_intensity'].mean():.4f}, min: {df_features['feat_lexical_polarity_intensity'].min():.4f}, max: {df_features['feat_lexical_polarity_intensity'].max():.4f}")

feat_lexical_polarity_intensity -> mean: 0.8031, min: -1.0000, max: 1.0000


#### Feature 18: `feat_nltk_clean_token_ratio`
Tỷ lệ số lượng từ thực chất (sau khi lọc stopwords nhưng bảo toàn từ phủ định) trên tổng số từ gốc trong `[0.0, 1.0]`. Đánh giá độ cô đọng ngữ nghĩa của văn bản đầu vào theo quy trình tiền xử lý NLTK.


In [42]:
def calc_feat_nltk_clean_token_ratio(rid):
    c = review_cache[rid]
    return float(round(min(len(c["clean_tokens"]) / max(len(c["raw_tokens"]), 1), 1.0), 4))

df_features["feat_nltk_clean_token_ratio"] = [calc_feat_nltk_clean_token_ratio(int(r)) for r in df_features["review_id"]]
print(f"feat_nltk_clean_token_ratio -> mean: {df_features['feat_nltk_clean_token_ratio'].mean():.4f}, min: {df_features['feat_nltk_clean_token_ratio'].min():.4f}, max: {df_features['feat_nltk_clean_token_ratio'].max():.4f}")

feat_nltk_clean_token_ratio -> mean: 0.5466, min: 0.3333, max: 0.9048


#### Feature 19: `feat_nltk_opinion_negation_density`
Mật độ các từ mang sắc thái quan điểm (Positive/Negative), từ phủ định và trạng từ nhấn mạnh trên tổng số từ gốc trong `[0.0, 1.0]`. Đo lường mức độ giàu biểu cảm của câu phục vụ phân tích cảm xúc 3 lớp.


In [43]:
def calc_feat_nltk_opinion_negation_density(rid):
    raw_toks = review_cache[rid]["raw_tokens"]
    if not raw_toks:
        return 0.0
    cnt = sum(1 for t in raw_toks if (t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG or t in NEGATION_AND_CONTRAST_WORDS or t in INTENSIFIERS))
    return float(round(min(cnt / len(raw_toks), 1.0), 4))

df_features["feat_nltk_opinion_negation_density"] = [calc_feat_nltk_opinion_negation_density(int(r)) for r in df_features["review_id"]]
print(f"feat_nltk_opinion_negation_density -> mean: {df_features['feat_nltk_opinion_negation_density'].mean():.4f}, max: {df_features['feat_nltk_opinion_negation_density'].max():.4f}")

feat_nltk_opinion_negation_density -> mean: 0.1156, max: 0.3889


#### Feature 20: `feat_ebr_aspect_prototype_sim`
Độ tương đồng trung bình Top-3 giữa tập từ khóa sạch của bài đánh giá và từ điển nguyên mẫu (prototype keywords) của 6 khía cạnh trong `[0.0, 1.0]`. Đóng vai trò tín hiệu neo ngữ nghĩa cho cơ chế truy hồi dựa trên embedding (EBR).


In [44]:
def calc_feat_ebr_aspect_prototype_sim(rid):
    clean_toks = review_cache[rid]["clean_tokens"]
    if not clean_toks:
        return 0.0
    tok_counts = Counter(clean_toks)
    sims = []
    for asp in CANONICAL_ASPECTS:
        kw_set = ASPECT_KEYWORDS[asp]
        overlap_w = sum(cnt for tok, cnt in tok_counts.items() if tok in kw_set)
        norm_sim = overlap_w / math.sqrt(len(clean_toks) * len(kw_set))
        sims.append(min(1.0, norm_sim * 3.5))
    top_sims = sorted(sims, reverse=True)[:3]
    return float(round(sum(top_sims) / len(top_sims), 4))

df_features["feat_ebr_aspect_prototype_sim"] = [calc_feat_ebr_aspect_prototype_sim(int(r)) for r in df_features["review_id"]]
print(f"feat_ebr_aspect_prototype_sim -> mean: {df_features['feat_ebr_aspect_prototype_sim'].mean():.4f}, min: {df_features['feat_ebr_aspect_prototype_sim'].min():.4f}, max: {df_features['feat_ebr_aspect_prototype_sim'].max():.4f}")

feat_ebr_aspect_prototype_sim -> mean: 0.3029, min: 0.0000, max: 0.7028


In [45]:
FEATURE_COLS_20 = [c for c in df_features.columns if c != "review_id"]
print(f"Đã trích xuất thành công {len(FEATURE_COLS_20)} features cho {len(df_features):,} bài đánh giá:")
print("Danh sách 20 features:", FEATURE_COLS_20)

cleaned_text_pairs = [preprocess_text_paper1(t) for t in df_reviews["review_text"]]
df_reviews["clean_review_text"] = [p[0] for p in cleaned_text_pairs]
df_reviews["paper1_nltk_text"] = [p[1] for p in cleaned_text_pairs]
df_merged = pd.merge(df_reviews, df_features, on="review_id")
display(df_features.head())

Đã trích xuất thành công 20 features cho 9,990 bài đánh giá:
Danh sách 20 features: ['feat_aspect_count', 'feat_dominant_aspect', 'feat_net_sentiment_score', 'feat_sentiment_conflict', 'feat_evidence_density', 'feat_avg_rationale_len', 'feat_has_cross_overlap', 'feat_service_sentiment', 'feat_facility_sentiment', 'feat_experience_sentiment', 'feat_amenity_sentiment', 'feat_loyalty_sentiment', 'feat_branding_sentiment', 'feat_aspect_coverage_ratio', 'feat_neutral_hedging_density', 'feat_consensus_strength', 'feat_lexical_polarity_intensity', 'feat_nltk_clean_token_ratio', 'feat_nltk_opinion_negation_density', 'feat_ebr_aspect_prototype_sim']


,review_id,feat_aspect_count,feat_dominant_aspect,feat_net_sentiment_score,feat_sentiment_conflict,feat_evidence_density,feat_avg_rationale_len,feat_has_cross_overlap,feat_service_sentiment,feat_facility_sentiment,feat_experience_sentiment,feat_amenity_sentiment,feat_loyalty_sentiment,feat_branding_sentiment,feat_aspect_coverage_ratio,feat_neutral_hedging_density,feat_consensus_strength,feat_lexical_polarity_intensity,feat_nltk_clean_token_ratio,feat_nltk_opinion_negation_density,feat_ebr_aspect_prototype_sim
0,1,3,1,0.6,1,0.5572,38.00,0,-0.6,1.0,1.0,0.0,0.0,0.0,0.5000,0.0149,0.6,-0.6000,0.4328,0.1791,0.4607
1,2,3,3,1.0,0,0.6107,68.57,0,1.0,1.0,1.0,0.0,0.0,0.0,0.5000,0.0077,1.0,0.7956,0.6462,0.0923,0.3807
2,3,4,3,1.0,0,0.5668,40.00,0,1.0,1.0,1.0,0.0,1.0,0.0,0.6667,0.0119,1.0,1.0000,0.5595,0.1429,0.2338
3,4,3,1,1.0,0,0.7238,65.50,0,1.0,1.0,0.0,0.0,1.0,0.0,0.5000,0.0000,1.0,1.0000,0.5052,0.0619,0.3262
4,5,2,2,1.0,0,0.8471,43.20,0,1.0,1.0,0.0,0.0,0.0,0.0,0.3333,0.0000,1.0,1.0000,0.6154,0.1538,0.3102


### 5.2 Lưu trữ DataFrame thành tệp Parquet và xuất CSDL


In [46]:
os.makedirs("data/df", exist_ok=True)

ann_grouped = df_annotations.groupby("review_id").apply(
    lambda g: g[["annotation_id", "aspect", "sentiment", "start_char", "end_char", "span_text"]].to_dict("records"),
    include_groups=False
).reset_index(name="aspect_annotations")

df_master = pd.merge(df_merged, ann_grouped, on="review_id", how="left")
df_master["aspect_annotations"] = df_master["aspect_annotations"].apply(lambda x: x if isinstance(x, list) else [])

parquet_path = "data/df/hotel_reviews.parquet"
df_master.to_parquet(parquet_path, compression="snappy", index=False)
print(f"Đã xuất tệp Parquet duy nhất của dự án tại: '{parquet_path}' ({len(df_master):,} dòng, {len(df_master.columns)} cột, {len(df_annotations):,} annotations lồng nhau).")

db_path = "data/df/chatbot.db"
df_features.to_sql("review_features", conn, if_exists="replace", index=False)
cursor.execute("CREATE INDEX IF NOT EXISTS idx_feat_review_id ON review_features(review_id);")
conn.commit()
print(f"Đã lưu bảng 'review_features' ({len(df_features):,} dòng) vào SQLite tại '{db_path}'.")

conn.close()
print("Hoàn tất lưu trữ dữ liệu trung gian.")

Đã xuất tệp Parquet duy nhất của dự án tại: 'data/df/hotel_reviews.parquet' (9,990 dòng, 38 cột, 54,326 annotations lồng nhau).
Đã lưu bảng 'review_features' (9,990 dòng) vào SQLite tại 'data/df/chatbot.db'.
Hoàn tất lưu trữ dữ liệu trung gian.


### 5.3 Tổng hợp Đa Đánh giá theo Đồng thuận Cảm xúc 3 Lớp (Positive, Neutral, Negative)


In [47]:
def demonstrate_multi_review_aspect_synthesis(df_master, hotel_target="Khách sạn pullman hà nội", aspect_target="Amenity", top_k=5):
    hotel_revs = df_master[df_master["hotel_name"].str.lower() == hotel_target.lower()].copy()
    if hotel_revs.empty:
        hotel_revs = df_master[df_master["hotel_name"].str.lower().str.contains("pullman", na=False)].copy()
    if hotel_revs.empty:
        print(f"Không tìm thấy review cho khách sạn: {hotel_target}")
        return

    actual_hotel_name = hotel_revs.iloc[0]["hotel_name"]

    def mentions_aspect(ann_list):
        if not hasattr(ann_list, "__iter__"):
            return False
        return any(a.get("aspect") == aspect_target for a in ann_list if isinstance(a, dict))

    def get_aspect_sentiment(ann_list):
        if not hasattr(ann_list, "__iter__"):
            return 0
        sents = [a.get("sentiment") for a in ann_list if isinstance(a, dict) and a.get("aspect") == aspect_target]
        if not sents:
            return 0
        pos, neg, neu = sents.count("Positive"), sents.count("Negative"), sents.count("Neutral")
        if pos > neg and pos >= neu:
            return 1
        if neg > pos and neg >= neu:
            return -1
        return 1 if pos > neg else (-1 if neg > pos else 0)

    def get_aspect_spans(ann_list):
        if not hasattr(ann_list, "__iter__"):
            return []
        return [a.get("span_text") for a in ann_list if isinstance(a, dict) and a.get("aspect") == aspect_target and a.get("span_text")]

    hotel_revs["has_aspect"] = hotel_revs["aspect_annotations"].apply(mentions_aspect)
    aspect_subset = hotel_revs[hotel_revs["has_aspect"]].copy()
    if aspect_subset.empty:
        aspect_subset = hotel_revs.copy()

    aspect_subset["aspect_sent_val"] = aspect_subset["aspect_annotations"].apply(get_aspect_sentiment)
    aspect_subset["evidence_spans"] = aspect_subset["aspect_annotations"].apply(get_aspect_spans)
    aspect_subset["evidence_weight"] = aspect_subset["feat_evidence_density"].clip(lower=0.05)

    top_candidates = aspect_subset.sort_values(by=["feat_evidence_density", "feat_consensus_strength"], ascending=False).head(top_k)

    weighted_sent = float(np.average(top_candidates["aspect_sent_val"], weights=top_candidates["evidence_weight"]))
    pos_pct = int(round((top_candidates["aspect_sent_val"] > 0).mean() * 100)) if len(top_candidates) > 0 else 0
    neu_pct = int(round((top_candidates["aspect_sent_val"] == 0).mean() * 100)) if len(top_candidates) > 0 else 0
    neg_pct = max(0, 100 - pos_pct - neu_pct)
    consensus_label = "Positive" if weighted_sent > 0.15 else ("Negative" if weighted_sent < -0.15 else "Neutral")

    print(f"THỰC NGHIỆM TRÍCH XUẤT ĐA REVIEW VÀ TỔNG HỢP ĐỒNG THUẬN 3 LỚP THEO KHÍA CẠNH: {aspect_target}")
    print(f"Khách sạn: {actual_hotel_name} | Số lượng reviews khớp khía cạnh: {len(top_candidates)} / {len(aspect_subset)}")
    print("Chi tiết các bằng chứng trích xuất theo mật độ dẫn chứng (Evidence Density):")
    for idx, (_, row) in enumerate(top_candidates.iterrows(), 1):
        sent_str = "Positive" if row["aspect_sent_val"] > 0 else ("Negative" if row["aspect_sent_val"] < 0 else "Neutral")
        span_sample = row["evidence_spans"][0] if row["evidence_spans"] else str(row.get("review_text", ""))[:90]
        print(f"  [{idx}] Review #{int(row['review_id'])} | Mật độ bằng chứng: {row['feat_evidence_density']*100:.1f}% | Cảm xúc: {sent_str}")
        print(f"      Dẫn chứng trích đoạn: '{span_sample[:95]}...'")

    print("Tổng hợp đồng thuận cảm xúc 3 lớp:")
    print(f"  - Phân bố 3 lớp: Positive {pos_pct}% | Neutral {neu_pct}% | Negative {neg_pct}%")
    print(f"  - Điểm đồng thuận có trọng số mật độ bằng chứng: {weighted_sent:+.3f} -> Kết luận: {consensus_label}")

demonstrate_multi_review_aspect_synthesis(df_master, hotel_target="Khách sạn pullman hà nội", aspect_target="Amenity")

THỰC NGHIỆM TRÍCH XUẤT ĐA REVIEW VÀ TỔNG HỢP ĐỒNG THUẬN 3 LỚP THEO KHÍA CẠNH: Amenity
Khách sạn: Khách sạn pullman hà nội | Số lượng reviews khớp khía cạnh: 5 / 6
Chi tiết các bằng chứng trích xuất theo mật độ dẫn chứng (Evidence Density):
  [1] Review #777 | Mật độ bằng chứng: 98.1% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Taxis readily available on call and that helps your commute...'
  [2] Review #2177 | Mật độ bằng chứng: 96.7% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'One can walk from Hotel to Ho Chi Minh Mausoleum within 10 minutes, you can also visit the One ...'
  [3] Review #1733 | Mật độ bằng chứng: 78.1% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Club Lounge excellent, good variety of hot and cold food, staff attentive...'
  [4] Review #2721 | Mật độ bằng chứng: 75.4% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Location very convenient...'
  [5] Review #3418 | Mật độ bằng chứng: 52.8% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Checking in

### 5.4 Tiền xử lý Văn bản và Phân chia Tập dữ liệu Huấn luyện


In [48]:
print("Bắt đầu tiền xử lý văn bản theo phương pháp paper 1.pdf và đồng bộ tệp hotel_reviews.parquet...")

parquet_path = "data/df/hotel_reviews.parquet"
df_proc = pd.read_parquet(parquet_path)

cleaned_pairs = [preprocess_text_paper1(txt) for txt in df_proc["review_text"]]
df_proc["clean_review_text"] = [p[0] for p in cleaned_pairs]
df_proc["paper1_nltk_text"] = [p[1] for p in cleaned_pairs]

def clean_annotation_spans(anns):
    if not isinstance(anns, (list, np.ndarray)):
        return []
    updated = []
    for ann in anns:
        if isinstance(ann, dict):
            item = dict(ann)
            c_span, nltk_span = preprocess_text_paper1(item.get("span_text", ""))
            item["clean_span_text"] = c_span
            item["paper1_span_tokens"] = nltk_span
            updated.append(item)
    return updated

df_proc["aspect_annotations"] = df_proc["aspect_annotations"].apply(clean_annotation_spans)

sample_paper1 = "I couldn't imagine a more beautiful place to stay. The staff were amazingly friendly. Could not fault the facilities and service. We didn't want to leave! We will be back for sure and won't stay anywhere else!"
sample_clean, sample_nltk = preprocess_text_paper1(sample_paper1)
print("Minh họa tiền xử lý văn bản theo paper 1.pdf (Table 4):")
print("  - Raw Text          :", sample_paper1)
print("  - clean_review_text :", sample_clean)
print("  - paper1_nltk_text  :", sample_nltk)

df_proc["corn_c0"] = (df_proc["sentiment_id"] > 0).astype(int)
df_proc["corn_c1"] = (df_proc["sentiment_id"] > 1).astype(int)

def clean_hotel_name(raw_name: str) -> str:
    if not raw_name:
        return "Hotel"
    text = str(raw_name).replace("Đ", "D").replace("đ", "d")
    norm = unicodedata.normalize("NFKD", text)
    ascii_text = "".join(c for c in norm if not unicodedata.combining(c))
    clean_str = ascii_text.replace("\xa0", " ").strip()
    clean_str = re.sub(r"^(khach\s*san|khu\s*nghi\s*duong|ks|hotel|resort|the)\s+", "", clean_str, flags=re.IGNORECASE).strip()
    clean_str = re.sub(r"\s*\.*$", "", clean_str).strip()
    return clean_str.title() if clean_str else "Hotel"

df_proc["clean_hotel_name"] = df_proc["hotel_name"].apply(clean_hotel_name)

sss_train = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_indices, temp_indices = next(sss_train.split(df_proc, df_proc["sentiment_id"]))

df_temp = df_proc.iloc[temp_indices]
sss_val_test = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_sub_idx, test_sub_idx = next(sss_val_test.split(df_temp, df_proc.iloc[temp_indices]["sentiment_id"]))
val_indices = temp_indices[val_sub_idx]
test_indices = temp_indices[test_sub_idx]

df_proc["split"] = "train"
df_proc.loc[val_indices, "split"] = "val"
df_proc.loc[test_indices, "split"] = "test"

df_proc.to_parquet(parquet_path, compression="snappy", index=False)

db_path = "data/df/chatbot.db"
conn = sqlite3.connect(db_path)
df_proc[["review_id", "clean_review_text", "paper1_nltk_text", "split", "overall_sentiment", "sentiment_id", "corn_c0", "corn_c1"]].to_sql(
    "train_split_info", conn, if_exists="replace", index=False
)
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_split ON train_split_info(split);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sent_id ON train_split_info(sentiment_id);")
conn.commit()
conn.close()

split_counts = df_proc["split"].value_counts()
print(f"\nĐã lưu tệp Parquet hoàn chỉnh tại: '{parquet_path}' ({len(df_proc):,} dòng, {len(df_proc.columns)} cột)")
print("Thống kê phân chia tập dữ liệu:")
for s_name in ["train", "val", "test"]:
    cnt = split_counts.get(s_name, 0)
    pct = (cnt / len(df_proc)) * 100
    print(f"  - Tập {s_name.upper():<5}: {cnt:,} bài đánh giá ({pct:.1f}%)")

print("\nPhân bố tỷ lệ nhãn cảm xúc 3 lớp (% trên từng tập sau khi Stratified Split):")
dist_table = pd.crosstab(df_proc["overall_sentiment"], df_proc["split"], normalize="columns").reindex(["Negative", "Neutral", "Positive"]) * 100
print(dist_table.round(2).to_string())

Bắt đầu tiền xử lý văn bản theo phương pháp paper 1.pdf và đồng bộ tệp hotel_reviews.parquet...
Minh họa tiền xử lý văn bản theo paper 1.pdf (Table 4):
  - Raw Text          : I couldn't imagine a more beautiful place to stay. The staff were amazingly friendly. Could not fault the facilities and service. We didn't want to leave! We will be back for sure and won't stay anywhere else!
  - clean_review_text : i could not imagine a more beautiful place to stay. the staff were amazingly friendly. could not fault the facilities and service. we did not want to leave! we will be back for sure and will not stay anywhere else!
  - paper1_nltk_text  : could not imagine beautiful place stay staff amazingly friendly could not fault facilities service not want leave back sure not stay anywhere else

Đã lưu tệp Parquet hoàn chỉnh tại: 'data/df/hotel_reviews.parquet' (9,990 dòng, 42 cột)
Thống kê phân chia tập dữ liệu:
  - Tập TRAIN: 7,992 bài đánh giá (80.0%)
  - Tập VAL  : 999 bài đánh giá (10.0%)
 